# NO₂ Data Cleaning and Validation (2020–2023)

This notebook cleans the raw hourly NO₂ dataset and converts it into a validated
city-day level dataset for further analysis and modeling.

### Cleaning Steps:
- Remove metadata rows
- Replace invalid values (-999)
- Convert date column
- Compute daily average from hourly values (H01–H24)
- Remove duplicates
- Handle missing values
- Add Year, Month, and Season features
- Export validated dataset



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../../data/raw")
files = sorted(RAW_DIR.glob("NO2_*.csv"))

df_list = []
for file in files:
    temp = pd.read_csv(file, skiprows=7)
    temp.columns = temp.columns.str.strip()
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print("Rows:", df.shape[0], "Cols:", df.shape[1])
df.head()


Rows: 286721 Cols: 31


,Pollutant//Polluant,NAPS ID//Identifiant SNPA,City//Ville,Province/Territory//Province/Territoire,Latitude//Latitude,Longitude//Longitude,Date//Date,H01//H01,H02//H02,H03//H03,...,H15//H15,H16//H16,H17//H17,H18//H18,H19//H19,H20//H20,H21//H21,H22//H22,H23//H23,H24//H24
0,NO2,10102,St. John's,NL,47.56038,-52.71147,1/1/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999
1,NO2,10102,St. John's,NL,47.56038,-52.71147,1/2/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999
2,NO2,10102,St. John's,NL,47.56038,-52.71147,1/3/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999
3,NO2,10102,St. John's,NL,47.56038,-52.71147,1/4/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999
4,NO2,10102,St. John's,NL,47.56038,-52.71147,1/5/2020,-999,-999,-999,...,-999,-999,-999,-999,-999,-999,-999,-999,-999,-999


In [2]:
hour_cols = [c for c in df.columns if "H" in c and "//" in c]
hour_cols = sorted(hour_cols)

print("Hourly columns:", len(hour_cols))
hour_cols[:5]

Hourly columns: 24


['H01//H01', 'H02//H02', 'H03//H03', 'H04//H04', 'H05//H05']

In [3]:
df.replace(-999, np.nan, inplace=True)

for c in hour_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")


In [4]:
df["NO2_daily_avg"] = df[hour_cols].mean(axis=1)

df[["NO2_daily_avg"]].describe()


,NO2_daily_avg
count,279486.000000
mean,5.953228
std,5.388991
min,0.000000
25%,2.166667
50%,4.375000
75%,8.083333
max,58.000000


In [5]:
df.columns

Index(['Pollutant//Polluant', 'NAPS ID//Identifiant SNPA', 'City//Ville',
       'Province/Territory//Province/Territoire', 'Latitude//Latitude',
       'Longitude//Longitude', 'Date//Date', 'H01//H01', 'H02//H02',
       'H03//H03', 'H04//H04', 'H05//H05', 'H06//H06', 'H07//H07', 'H08//H08',
       'H09//H09', 'H10//H10', 'H11//H11', 'H12//H12', 'H13//H13', 'H14//H14',
       'H15//H15', 'H16//H16', 'H17//H17', 'H18//H18', 'H19//H19', 'H20//H20',
       'H21//H21', 'H22//H22', 'H23//H23', 'H24//H24', 'NO2_daily_avg'],
      dtype='object')

In [6]:
# Select required columns
final = df[["Date//Date", "City//Ville", "NO2_daily_avg"]].copy()

# Rename columns to clean names
final.rename(columns={
    "Date//Date": "Date",
    "City//Ville": "City"
}, inplace=True)

# Remove duplicates
final.drop_duplicates(subset=["Date", "City"], inplace=True)

# Remove missing daily averages
final.dropna(subset=["NO2_daily_avg"], inplace=True)

final.head()


,Date,City,NO2_daily_avg
48,2/18/2020,St. John's,2.416667
49,2/19/2020,St. John's,1.875000
50,2/20/2020,St. John's,0.958333
51,2/21/2020,St. John's,0.750000
52,2/22/2020,St. John's,1.500000


In [7]:
final["Date"] = pd.to_datetime(final["Date"], errors="coerce")

final["Year"] = final["Date"].dt.year
final["Month"] = final["Date"].dt.month

def season_from_month(m):
    if m in [12, 1, 2]:
        return "Winter"
    elif m in [3, 4, 5]:
        return "Spring"
    elif m in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

final["Season"] = final["Month"].apply(season_from_month)

final.head()


,Date,City,NO2_daily_avg,Year,Month,Season
48,2020-02-18,St. John's,2.416667,2020.0,2.0,Winter
49,2020-02-19,St. John's,1.875000,2020.0,2.0,Winter
50,2020-02-20,St. John's,0.958333,2020.0,2.0,Winter
51,2020-02-21,St. John's,0.750000,2020.0,2.0,Winter
52,2020-02-22,St. John's,1.500000,2020.0,2.0,Winter


In [8]:
from pathlib import Path

OUT_DIR = Path("../../data/validated")
OUT_DIR.mkdir(parents=True, exist_ok=True)

final.to_csv(OUT_DIR / "NO2_cityday.csv", index=False)

print("NO2 cleaned dataset saved successfully.")
print("Final shape:", final.shape)


NO2 cleaned dataset saved successfully.
Final shape: (217619, 6)
